In [0]:
# Load all cleaned Silver tables for validation

villages_df = spark.table("workspace.agri_intelligence.silver_villages")
farmers_df = spark.table("workspace.agri_intelligence.silver_farmers")
products_df = spark.table("workspace.agri_intelligence.silver_products")
bills_df = spark.table("workspace.agri_intelligence.silver_bills")
bill_items_df = spark.table("workspace.agri_intelligence.silver_bill_items")

print("All Silver tables loaded successfully.")

All Silver tables loaded successfully.


In [0]:
# Check duplicate primary keys in all five tables

villages_df.groupBy("village_id").count().filter("count > 1").show()
farmers_df.groupBy("farmer_id").count().filter("count > 1").show()
products_df.groupBy("product_id").count().filter("count > 1").show()
bills_df.groupBy("bill_id").count().filter("count > 1").show()
bill_items_df.groupBy("bill_item_id").count().filter("count > 1").show()

+----------+-----+
|village_id|count|
+----------+-----+
+----------+-----+

+---------+-----+
|farmer_id|count|
+---------+-----+
+---------+-----+

+----------+-----+
|product_id|count|
+----------+-----+
+----------+-----+

+-------+-----+
|bill_id|count|
+-------+-----+
+-------+-----+

+------------+-----+
|bill_item_id|count|
+------------+-----+
+------------+-----+



In [0]:
from pyspark.sql.functions import col, trim

# Check missing IDs in all five tables

print("Missing village IDs:",
      villages_df.filter(col("village_id").isNull() | (trim(col("village_id")) == "")).count())

print("Missing farmer IDs:",
      farmers_df.filter(col("farmer_id").isNull() | (trim(col("farmer_id")) == "")).count())

print("Missing product IDs:",
      products_df.filter(col("product_id").isNull() | (trim(col("product_id")) == "")).count())

print("Missing bill IDs:",
      bills_df.filter(col("bill_id").isNull() | (trim(col("bill_id")) == "")).count())

print("Missing bill item IDs:",
      bill_items_df.filter(col("bill_item_id").isNull() | (trim(col("bill_item_id")) == "")).count())

Missing village IDs: 0
Missing farmer IDs: 0
Missing product IDs: 0
Missing bill IDs: 0
Missing bill item IDs: 0


In [0]:
# Find farmers whose village does not exist

farmers_df.join(
    villages_df,
    "village_id",
    "left_anti"
).show()

+----------+---------+-----------+-------------+------------+
|village_id|farmer_id|farmer_name|mobile_number|village_name|
+----------+---------+-----------+-------------+------------+
+----------+---------+-----------+-------------+------------+



In [0]:
# Find bills with farmer IDs that do not exist

bills_df.filter(col("farmer_id").isNotNull()).join(
    farmers_df,
    "farmer_id",
    "left_anti"
).show()

+---------+-------+-----------+---------+----------------------+----------------------+-------------------+---------------------+---------------+--------------+------------------------+
|farmer_id|bill_id|bill_number|bill_date|customer_name_snapshot|mobile_number_snapshot|village_id_snapshot|village_name_snapshot|item_line_count|bill_total_inr|customer_identity_status|
+---------+-------+-----------+---------+----------------------+----------------------+-------------------+---------------------+---------------+--------------+------------------------+
+---------+-------+-----------+---------+----------------------+----------------------+-------------------+---------------------+---------------+--------------+------------------------+



In [0]:
# Find bill items whose bill does not exist

bill_items_df.join(
    bills_df,
    "bill_id",
    "left_anti"
).show()

+-------+------------+----------+--------------------+----+--------------+-------------+--------------+
|bill_id|bill_item_id|product_id|product_display_name|crop|quantity_packs|unit_rate_inr|line_total_inr|
+-------+------------+----------+--------------------+----+--------------+-------------+--------------+
+-------+------------+----------+--------------------+----+--------------+-------------+--------------+



In [0]:
# Find bill items whose product does not exist

bill_items_df.join(
    products_df,
    "product_id",
    "left_anti"
).show()

+----------+------------+-------+--------------------+----+--------------+-------------+--------------+
|product_id|bill_item_id|bill_id|product_display_name|crop|quantity_packs|unit_rate_inr|line_total_inr|
+----------+------------+-------+--------------------+----+--------------+-------------+--------------+
+----------+------------+-------+--------------------+----+--------------+-------------+--------------+



In [0]:
# Check the village names in our dataset

villages_df.select("village_name", "normalised_village_name").show(20, False)

# Count total and unique villages

print("Total villages:", villages_df.count())

print("Unique village names:",
      villages_df.select("normalised_village_name").distinct().count())

+------------+-----------------------+
|village_name|normalised_village_name|
+------------+-----------------------+
|Modepalli   |modepalli              |
|Kotikalapudi|kotikalapudi           |
|Rallapalli  |rallapalli             |
|Kalavakuru  |kalavakuru             |
|Kunkupadu   |kunkupadu              |
|Vemparala   |vemparala              |
|Kopperapadu |kopperapadu            |
+------------+-----------------------+

Total villages: 7
Unique village names: 7


In [0]:
# Compare actual villages with our approved village list

expected_villages = [
    "modepalli",
    "kotikalapudi",
    "rallapalli",
    "kalavakuru",
    "kunkupadu",
    "vemparala",
    "kopperapadu"
]

actual_villages = [
    row["normalised_village_name"]
    for row in villages_df.select("normalised_village_name").collect()
]

print("Missing villages:", set(expected_villages) - set(actual_villages))
print("Unexpected villages:", set(actual_villages) - set(expected_villages))

Missing villages: set()
Unexpected villages: set()


In [0]:
# Check whether quantity multiplied by unit rate matches line total

invalid_items = bill_items_df.filter(
    col("line_total_inr") !=
    (col("quantity_packs") * col("unit_rate_inr"))
)

print("Incorrect bill items:", invalid_items.count())

invalid_items.show(10)

Incorrect bill items: 0
+------------+-------+----------+--------------------+----+--------------+-------------+--------------+
|bill_item_id|bill_id|product_id|product_display_name|crop|quantity_packs|unit_rate_inr|line_total_inr|
+------------+-------+----------+--------------------+----+--------------+-------------+--------------+
+------------+-------+----------+--------------------+----+--------------+-------------+--------------+



In [0]:
from pyspark.sql.functions import col, sum as spark_sum, count as spark_count

# Calculate total amount and item count for each bill

bill_item_totals = bill_items_df.groupBy("bill_id").agg(
    spark_sum(col("line_total_inr").cast("decimal(12,2)")).alias("calculated_total"),
    spark_count("bill_item_id").alias("calculated_item_count")
)

bill_item_totals.show(10)

+---------+----------------+---------------------+
|  bill_id|calculated_total|calculated_item_count|
+---------+----------------+---------------------+
|BILL00001|         3866.00|                    2|
|BILL00002|         6755.00|                    2|
|BILL00003|         3653.00|                    2|
|BILL00004|         6363.00|                    3|
|BILL00005|         2133.00|                    1|
|BILL00006|         1179.00|                    2|
|BILL00007|          670.00|                    1|
|BILL00008|         1802.00|                    1|
|BILL00009|         2607.00|                    2|
|BILL00010|         2619.00|                    2|
+---------+----------------+---------------------+
only showing top 10 rows


In [0]:
# Compare calculated totals with bill header records

bill_check = bills_df.join(
    bill_item_totals,
    "bill_id",
    "left"
)

bill_check = bill_check.filter(
    (col("bill_total_inr").cast("decimal(12,2)") != col("calculated_total")) |
    (col("item_line_count").cast("int") != col("calculated_item_count")) |
    col("calculated_total").isNull()
)

print("Bills with reconciliation issues:", bill_check.count())

bill_check.show(10)

Bills with reconciliation issues: 0
+-------+-----------+---------+---------+----------------------+----------------------+-------------------+---------------------+---------------+--------------+------------------------+----------------+---------------------+
|bill_id|bill_number|bill_date|farmer_id|customer_name_snapshot|mobile_number_snapshot|village_id_snapshot|village_name_snapshot|item_line_count|bill_total_inr|customer_identity_status|calculated_total|calculated_item_count|
+-------+-----------+---------+---------+----------------------+----------------------+-------------------+---------------------+---------------+--------------+------------------------+----------------+---------------------+
+-------+-----------+---------+---------+----------------------+----------------------+-------------------+---------------------+---------------+--------------+------------------------+----------------+---------------------+

